# Model Parameters

> Parameter dataclasses for the Rosen-Roback spatial equilibrium model


In [1]:
# | default_exp parameters

In [2]:
# | export
from dataclasses import dataclass, field
from typing import ClassVar

## Model Parameters

The Rosen-Roback model has four key parameters that control how the spatial equilibrium behaves:

### α (alpha): Labour Share in Production

- Determines how output is split between labour and land/capital
- Standard estimates: 0.6-0.7 (workers get 60-70% of output)
- Example: If α=0.65, then a 1% increase in employment increases output by 0.65%

### η (eta): Agglomeration Elasticity

- Controls how much productivity increases when a city gets larger
- This is the **key channel** for why big cities are more productive
- Standard estimates: 0.02-0.08 (2-8% productivity boost from doubling city size)
- Cooped Up uses 0.25 (very strong agglomeration!)

### β (beta): Housing Expenditure Share

- Fraction of income spent on housing
- Standard estimate: 0.33 (one-third of income)
- Higher β means workers care more about rents when choosing where to live

### θ (theta): Labour Mobility (Fréchet Shape Parameter)

- Controls how much utility spread exists across cities in equilibrium
- Higher θ → workers more responsive to utility differences → smaller utility spread
- θ=3: ~15% utility spread (significant moving costs/preferences)
- θ=10: ~6% utility spread (moderate mobility)
- θ=50+: <2% spread (classic Rosen-Roback with nearly equal utilities)

### Why Theta Matters

In the classic Rosen-Roback model, workers are perfectly mobile and utilities are exactly equal across cities. In reality, people face moving costs and have location preferences. The Fréchet framework (from trade models) captures this:

- Low θ: Workers have strong location preferences → utilities can differ substantially
- High θ: Workers are very mobile → utilities converge (classic case)

This matters for counterfactuals: if θ is low, relaxing housing supply in one city won't cause massive population shifts because people really like their current locations.


In [ ]:
# | export
@dataclass(frozen=True)
class ModelParameters:
    """Global parameters for the Rosen-Roback spatial equilibrium model."""

    alpha: float = 0.65  # Labour share in production (typically 0.6-0.7)
    eta: float = 0.04  # Agglomeration elasticity (standard: 0.02-0.08, Cooped Up: 0.25)
    beta: float = 0.33  # Housing expenditure share (fraction of income spent on housing)
    theta: float = 10.0  # Labour mobility (Fréchet shape). Higher = more mobile workers

    # Named parameter sets
    STANDARD: ClassVar["ModelParameters"]
    HSIEH_MORETTI: ClassVar["ModelParameters"]
    COOPED_UP: ClassVar["ModelParameters"]
    HIGH_MOBILITY: ClassVar["ModelParameters"]

    def __post_init__(self) -> None:
        """Validate parameters are in reasonable ranges."""
        if not 0 < self.alpha < 1:
            raise ValueError(f"alpha must be in (0, 1), got {self.alpha}")
        if self.eta < 0:
            raise ValueError(f"eta must be non-negative, got {self.eta}")
        if not 0 < self.beta < 1:
            raise ValueError(f"beta must be in (0, 1), got {self.beta}")
        if self.theta <= 0:
            raise ValueError(f"theta must be positive, got {self.theta}")

    def with_updates(self, **kwargs) -> "ModelParameters":
        """Create a new ModelParameters with some values updated."""
        return ModelParameters(
            alpha=kwargs.get("alpha", self.alpha),
            eta=kwargs.get("eta", self.eta),
            beta=kwargs.get("beta", self.beta),
            theta=kwargs.get("theta", self.theta),
        )

### Named Parameter Sets

We provide several pre-calibrated parameter sets based on published research:


In [11]:
# | export
# Standard academic estimates with moderate mobility
ModelParameters.STANDARD = ModelParameters(
    alpha=0.65,
    eta=0.04,
    beta=0.33,
    theta=10.0,  # ~6% utility spread
)

# Hsieh-Moretti (2019) calibration
ModelParameters.HSIEH_MORETTI = ModelParameters(
    alpha=0.65,
    eta=0.25,
    beta=0.33,
    theta=2.0,  # Original paper value
)

# Cooped Up paper calibration
ModelParameters.COOPED_UP = ModelParameters(
    alpha=0.65,
    eta=0.25,
    beta=0.32,
    theta=3.33,  # Original paper value
)

# High mobility (classic Rosen-Roback with nearly equal utilities)
ModelParameters.HIGH_MOBILITY = ModelParameters(
    alpha=0.65,
    eta=0.04,
    beta=0.33,
    theta=50.0,  # ~1.5% utility spread
)

## City Parameters

Each city in the model has four characteristics:

### Base TFP (Total Factor Productivity)

- Productivity before agglomeration effects
- Higher TFP cities attract more workers (if housing supply allows)
- In equilibrium, agglomeration amplifies TFP differences

### Amenity

- How much workers like living in this city, independent of wages and rents
- Could capture: weather, culture, family ties, crime, pollution
- In classic Rosen-Roback, amenities are **calibrated** to rationalize observed populations

### Supply Elasticity (γ, gamma)

- How much housing supply increases when rents rise
- γ = % increase in housing / % increase in rent
- Houston: ~2.5 (very elastic, flat land)
- San Francisco: ~0.7 (inelastic, geography + regulations)
- **This is the key policy parameter!**

### Supply Shifter

- Determines the level of rents at any given population
- Can represent land availability or construction costs
- Usually calibrated to match observed rents


In [ ]:
# | export
@dataclass
class CityParameters:
    """Parameters for a single city in the model."""

    name: str  # City identifier (e.g., "London", "Manchester")
    base_tfp: float  # Base total factor productivity (before agglomeration)
    amenity: float = 1.0  # Amenity value (how much workers like this city)
    supply_elasticity: float = 2.0  # Housing supply elasticity (gamma). SF ~0.7, Houston ~2.5
    supply_shifter: float = 1000.0  # Housing supply shifter (affects rent level)

    def __post_init__(self) -> None:
        """Validate city parameters."""
        if self.base_tfp <= 0:
            raise ValueError(f"base_tfp must be positive, got {self.base_tfp}")
        if self.amenity <= 0:
            raise ValueError(f"amenity must be positive, got {self.amenity}")
        if self.supply_elasticity <= 0:
            raise ValueError(f"supply_elasticity must be positive, got {self.supply_elasticity}")
        if self.supply_shifter <= 0:
            raise ValueError(f"supply_shifter must be positive, got {self.supply_shifter}")

## Equilibrium Results

The `EquilibriumResult` dataclass stores the solution to the spatial equilibrium problem.


In [ ]:
# | export
@dataclass
class EquilibriumResult:
    """Results from solving the spatial equilibrium."""

    city_names: list[str]  # List of city names in order
    population: list[float] = field(default_factory=list)  # Equilibrium population in each city
    wages: list[float] = field(default_factory=list)  # Equilibrium wage in each city
    rents: list[float] = field(default_factory=list)  # Equilibrium rent in each city
    utilities: list[float] = field(default_factory=list)  # Worker utility in each city
    outputs: list[float] = field(default_factory=list)  # Total output produced in each city
    total_gdp: float = 0.0  # Sum of outputs across all cities
    converged: bool = False  # Whether the solver converged
    iterations: int = 0  # Number of iterations taken

    def __len__(self) -> int:
        """Number of cities."""
        return len(self.city_names)

    def to_dataframe(self):
        """Convert results to a pandas DataFrame."""
        import pandas as pd

        return pd.DataFrame(
            {
                "city": self.city_names,
                "population": self.population,
                "wage": self.wages,
                "rent": self.rents,
                "utility": self.utilities,
                "output": self.outputs,
            }
        )

    def utility_spread_pct(self) -> float:
        """Calculate percentage spread in utilities (max/min - 1)."""
        if not self.utilities:
            return 0.0
        utils = [u for u in self.utilities if u > 0]
        if not utils:
            return 0.0
        return (max(utils) / min(utils) - 1) * 100

    def summary(self) -> str:
        """Return a formatted summary string."""
        lines = [
            "Spatial Equilibrium Results",
            "=" * 70,
            f"{'City':<25} {'Population':>12} {'Wage':>10} {'Rent':>10} {'Utility':>10}",
            "-" * 70,
        ]
        for i, name in enumerate(self.city_names):
            lines.append(
                f"{name:<25} {self.population[i]:>12,.0f} "
                f"{self.wages[i]:>10.2f} {self.rents[i]:>10.2f} "
                f"{self.utilities[i]:>10.4f}"
            )
        lines.append("-" * 70)
        lines.append(f"Total GDP: {self.total_gdp:,.0f}")
        spread = self.utility_spread_pct()
        lines.append(f"Utility spread: {spread:.1f}% (smaller = closer to classic R-R equilibrium)")
        lines.append(f"Converged: {self.converged} ({self.iterations} iterations)")
        return "\n".join(lines)

## Tests


In [7]:
# | hide
# Test ModelParameters validation
params = ModelParameters()  # Should work with defaults
assert 0 < params.alpha < 1
assert params.eta >= 0
assert 0 < params.beta < 1
assert params.theta > 0

# Test invalid parameters
try:
    ModelParameters(alpha=1.5)  # Should fail
    assert False, "Should have raised ValueError"
except ValueError:
    pass

# Test named parameter sets
assert ModelParameters.STANDARD.eta == 0.04
assert ModelParameters.HSIEH_MORETTI.eta == 0.25
assert ModelParameters.COOPED_UP.theta == 3.33
assert ModelParameters.HIGH_MOBILITY.theta == 50.0

# Test with_updates
updated = params.with_updates(eta=0.06)
assert updated.eta == 0.06
assert updated.alpha == params.alpha  # Other params unchanged

# Test CityParameters
london = CityParameters(name="London", base_tfp=120)
assert london.name == "London"
assert london.amenity == 1.0  # Default

# Test EquilibriumResult
result = EquilibriumResult(city_names=["A", "B"])
assert len(result) == 2

## Example Usage


In [8]:
# Compare different parameter sets
print("Standard parameters:")
print(f"  Agglomeration (eta): {ModelParameters.STANDARD.eta}")
print(f"  Mobility (theta): {ModelParameters.STANDARD.theta}")
print()
print("Hsieh-Moretti parameters:")
print(f"  Agglomeration (eta): {ModelParameters.HSIEH_MORETTI.eta}")
print(f"  Mobility (theta): {ModelParameters.HSIEH_MORETTI.theta}")
print()
print("Key difference: Hsieh-Moretti has MUCH stronger agglomeration (0.25 vs 0.04)")
print("This means big productivity gains from city size → large GDP losses from misallocation")

Standard parameters:
  Agglomeration (eta): 0.04
  Mobility (theta): 10.0

Hsieh-Moretti parameters:
  Agglomeration (eta): 0.25
  Mobility (theta): 2.0

Key difference: Hsieh-Moretti has MUCH stronger agglomeration (0.25 vs 0.04)
This means big productivity gains from city size → large GDP losses from misallocation


In [9]:
# Create a simple two-city example
productive_city = CityParameters(
    name="HighTech City",
    base_tfp=120,
    supply_elasticity=0.7,  # Constrained like SF
)

normal_city = CityParameters(
    name="Normal City",
    base_tfp=100,
    supply_elasticity=2.5,  # Elastic like Houston
)

print(f"{productive_city.name}: TFP={productive_city.base_tfp}, Supply Elasticity={productive_city.supply_elasticity}")
print(f"{normal_city.name}: TFP={normal_city.base_tfp}, Supply Elasticity={normal_city.supply_elasticity}")
print()
print("Prediction: HighTech City will have higher wages AND rents.")
print("The housing supply constraint (low elasticity) limits population growth.")

HighTech City: TFP=120, Supply Elasticity=0.7
Normal City: TFP=100, Supply Elasticity=2.5

Prediction: HighTech City will have higher wages AND rents.
The housing supply constraint (low elasticity) limits population growth.


In [10]:
# | hide
import nbdev

nbdev.nbdev_export()